# RAG-Praxis: Retrieval-Augmented Generation (lokal)

Dieses Notebook führt den **RAG-Ablauf** Schritt für Schritt aus – **vollständig lokal**, ohne externe LLM-API. Sie lernen:

1. Einen kleinen Dokumentenkorpus zu chunken und zu embedden
2. Semantische Suche (Retrieval) mit Kosinusähnlichkeit
3. Den RAG-Prompt zu bauen (Kontext + Frage)
4. Eine lokale „Antwort“ ohne API (Mock-Generierung auf Basis des Kontexts)

**Voraussetzung:** ``sentence-transformers`` (z. B. ``pip install sentence-transformers``). Beim ersten Lauf wird ein kleines Embedding-Modell heruntergeladen, danach läuft alles offline.

## 1. Abhängigkeiten und Embedding-Modell

In [ ]:
import numpy as np

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    raise ImportError("Bitte installieren: pip install sentence-transformers")

# Kleines, schnelles Modell – läuft gut auf CPU
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding-Modell geladen.")

## 2. Beispiel-Dokumente und Chunking

Wir verwenden einen kleinen deutschen Korpus aus Absätzen. In der Praxis ersetzen Sie ihn durch Ihre eigenen Dokumente (z. B. FAQs, Handbücher).

In [ ]:
docs = [
    "RAG bedeutet Retrieval-Augmented Generation. Man sucht zuerst relevante Textstellen in eigenen Dokumenten und gibt sie dem Sprachmodell als Kontext mit.",
    "Embeddings sind Vektoren, die die Bedeutung von Text erfassen. Ähnliche Texte haben ähnliche Vektoren und liegen im Vektorraum nah beieinander.",
    "Bei der semantischen Suche wird nicht nach exakten Wörtern gesucht, sondern nach inhaltlich passenden Stellen. Dafür nutzt man Embeddings und ein Ähnlichkeitsmaß wie die Kosinusähnlichkeit.",
    "Ein Large Language Model erzeugt Text Token für Token. Es sagt das nächste Wort vorher und fügt es an den bisherigen Kontext an.",
    "Ohne RAG kennt das Modell nur sein Trainingswissen. Mit RAG kann es auf aktuelle Dokumente und firmeneigenes Wissen zugreifen.",
]

# Einfaches Chunking: hier jeder Absatz = ein Chunk
chunks = docs
print(f"Anzahl Chunks: {len(chunks)}")

## 3. Embeddings berechnen und speichern

In [ ]:
chunk_embeddings = model.encode(chunks)
chunk_embeddings = np.array(chunk_embeddings)
print(f"Shape der Embeddings: {chunk_embeddings.shape}")

## 4. Retrieval: Kosinusähnlichkeit und Top-k

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query: str, top_k: int = 2):
    q_emb = model.encode([query])[0]
    scores = [cosine_similarity(q_emb, c) for c in chunk_embeddings]
    idx = np.argsort(scores)[::-1][:top_k]
    return [(i, scores[i], chunks[i]) for i in idx]

# Testabfrage
query = "Wie funktioniert RAG?"
results = retrieve(query, top_k=2)
print("Top-2 Chunks zur Frage:", repr(query))
for i, (idx, score, text) in enumerate(results, 1):
    print(f"  {i}. (Score {score:.3f}) {text[:80]}...")

## 5. RAG-Prompt bauen

Der Prompt enthält den **Kontext** (die abgerufenen Chunks) und die **Frage**. Ein echtes LLM würde daraus eine Antwort generieren. Hier bauen wir nur den Prompt – ohne API-Aufruf.

In [ ]:
def build_rag_prompt(query: str, retrieved_chunks: list, max_context_len: int = 500):
    context = "\n".join(r[2] for r in retrieved_chunks)
    if len(context) > max_context_len:
        context = context[:max_context_len] + "..."
    return (
        "Antworte nur auf Basis des folgenden Kontexts. "
        "Wenn die Antwort nicht im Kontext steht, sage das.\n\n"
        "Kontext:\n" + context + "\n\n"
        "Frage: " + query + "\n\n"
        "Antwort:"
    )

rag_prompt = build_rag_prompt(query, results)
print("=== RAG-Prompt (würde an ein LLM gesendet) ===")
print(rag_prompt)

## 6. Lokale „Antwort“ ohne externe API

Damit das Notebook **ohne LLM-API** auskommt, erzeugen wir eine **Mock-Antwort**: Sie fasst die abgerufenen Chunks zusammen. In der Praxis ersetzen Sie diesen Schritt durch einen Aufruf an ein lokales LLM (z. B. Ollama) oder eine Cloud-API.

In [ ]:
def mock_answer(query: str, retrieved_chunks: list) -> str:
    """Erzeugt eine lokale Antwort ohne API: Kurze Zusammenfassung der Chunks."""
    parts = [r[2] for r in retrieved_chunks]
    summary = " ".join(parts)
    if len(summary) > 400:
        summary = summary[:400] + "..."
    return (
        "[Lokale Mock-Antwort – ohne LLM-API]\n"
        "Basierend auf dem abgerufenen Kontext:\n\n" + summary
    )

answer = mock_answer(query, results)
print(answer)

## 7. Andere Fragen ausprobieren

In [ ]:
for q in ["Was sind Embeddings?", "Wozu braucht man semantische Suche?"]:
    res = retrieve(q, top_k=2)
    ans = mock_answer(q, res)
    print("Frage:", q)
    print(ans[:300], "..." if len(ans) > 300 else "")
    print("-" * 50)

## Zusammenfassung

* **Embeddings** und **Retrieval** laufen vollständig lokal (sentence-transformers).
* Der **RAG-Prompt** (Kontext + Frage) kann bei Bedarf an ein lokales LLM (z. B. Ollama) oder eine API gesendet werden.
* Die **Mock-Antwort** zeigt den Ablauf ohne externe API; in der echten Anwendung ersetzen Sie sie durch den LLM-Aufruf.